# 实验二 A：线性回归建模与参数可视化分析

**实验目的**

1. 通过 $y = wx + b + \mathrm{randn}$ 构造随机数据 $\{x, y\}$，掌握监督学习中回归模型的基本建模方法；
2. 引入新的参数 $w_1, b_1$ 进行训练拟合，观察模型收敛过程，并通过可视化展示**参数变化轨迹**与**拟合效果**；
3. 扩展数据规模并划分训练集与测试集，掌握模型**泛化能力**的评估方法。

> 说明：本实验的建模对象是标准**线性回归**（平方损失 + 小批量随机梯度下降）。
> 为与课件记号一致，把待训练的参数记为 $w_1, b_1$，真实（生成数据所用）参数记为 $w, b$。
> 全部代码“从零实现”，不调用 `torch.nn` / `torch.optim` 中的现成封装。

In [1]:
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = "plotly_white"

SEED = 42
torch.manual_seed(SEED)

# 配色：分类色按固定顺序取用；真实值/参考线统一用中性深灰，估计值统一用蓝
C_BLUE, C_ORANGE, C_RED = "#2a78d6", "#eb6834", "#e34948"
C_TRUE, C_DATA, C_MUTED = "#0b0b0b", "#8a8a86", "#52514e"
# 蓝色顺序色阶（浅 -> 深），用于损失等高线背景
BLUES = [[0.0, "#cde2fb"], [0.2, "#9ec5f4"], [0.4, "#6da7ec"],
         [0.6, "#3987e5"], [0.8, "#256abf"], [1.0, "#0d366b"]]


def style(fig, title="", xlabel="", ylabel="", width=760, height=430, legend_bottom=True):
    """统一的图形样式：标题、轴标签、底部图例、留白"""
    fig.update_layout(
        title=dict(text=title, x=0.02, xanchor="left"),
        xaxis_title=xlabel,
        yaxis_title=ylabel,
        width=width,
        height=height,
        margin=dict(l=70, r=30, t=70, b=80),
        legend=dict(orientation="h", yanchor="top", y=-0.18, x=0, title=None),
        hovermode="closest",
    )
    if not legend_bottom:
        fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0, title=None))
    return fig


print("torch", torch.__version__, "| numpy", np.__version__, "| plotly", pio.templates.default)

torch 2.13.0 | numpy 2.4.6 | plotly plotly_white


## 一、构造随机数据 $y = wx + b + \mathrm{randn}$

给定真实参数 $w, b$，按
$$x \sim \mathcal{N}(0, 1), \qquad y = wx + b + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2)$$
生成 $n$ 个样本 $\{x, y\}$。这里的 $\varepsilon$ 即题目中的 `randn` 噪声项，
它决定了模型**无论如何训练都无法消除**的误差下限。

In [2]:
TRUE_W, TRUE_B = 2.0, 4.2     # 真实参数（生成数据所用）
NOISE_STD = 0.6              # 噪声标准差 sigma


def synthetic_data(w, b, num_examples, noise_std=NOISE_STD, seed=SEED):
    """构造 y = w x + b + randn 数据"""
    rng = np.random.default_rng(seed)
    x = rng.normal(0.0, 1.0, size=(num_examples, 1))
    y = w * x + b + rng.normal(0.0, noise_std, size=(num_examples, 1))   # 噪声项 randn
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


N = 100                                     # 第一阶段的样本量
X, y = synthetic_data(TRUE_W, TRUE_B, N)

print(f"特征 X: {tuple(X.shape)}   标签 y: {tuple(y.shape)}")
print(f"真实模型: y = {TRUE_W} x + {TRUE_B} + eps,  eps ~ N(0, {NOISE_STD ** 2})")
print(f"x 均值 {X.mean():+.3f}   y 均值 {y.mean():+.3f}   y 标准差 {y.std():.3f}")
print("前 5 个样本：")
for xi, yi in zip(X[:5].flatten().tolist(), y[:5].flatten().tolist()):
    print(f"   x = {xi:+.4f}  ->  y = {yi:+.4f}")

特征 X: (100, 1)   标签 y: (100, 1)
真实模型: y = 2.0 x + 4.2 + eps,  eps ~ N(0, 0.36)
x 均值 -0.050   y 均值 +4.093   y 标准差 1.712
前 5 个样本：
   x = +0.3047  ->  y = +4.5825
   x = -1.0400  ->  y = +2.8996
   x = +0.7505  ->  y = +5.4871
   x = +0.9406  ->  y = +6.5236
   x = -1.9510  ->  y = -0.2622


In [3]:
Xn, yn = X.flatten().numpy(), y.flatten().numpy()


def line_xy(x, w, b, pad=0.2):
    """在当前 x 的取值范围内作直线 y = w x + b，供绘图使用"""
    xs = np.linspace(x.min() - pad, x.max() + pad, 100)
    return xs, w * xs + b


fig = go.Figure()
fig.add_trace(go.Scatter(
    x=Xn, y=yn, mode="markers", name="观测样本 (x, y)",
    marker=dict(size=7, color=C_DATA, opacity=0.75, line=dict(width=0.5, color="#fcfcfb")),
    hovertemplate="x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>",
))
xs, ys = line_xy(Xn, TRUE_W, TRUE_B)
fig.add_trace(go.Scatter(
    x=xs, y=ys, mode="lines", name=f"真实直线 y = {TRUE_W}x + {TRUE_B}",
    line=dict(color=C_TRUE, width=2, dash="dash"), hoverinfo="skip",
))
style(fig, f"随机数据与真实直线（n = {N}）", "x", "y")
fig.show()

## 二、从零实现：新参数 $w_1, b_1$ 的训练与收敛过程

线性回归的三个要素：

| 要素 | 形式 |
|---|---|
| 模型 | $\hat{y} = w_1 x + b_1$ |
| 损失 | $\ell(w_1, b_1) = \frac{1}{2}(\hat{y} - y)^2$，小批量上取均值 |
| 优化 | 小批量随机梯度下降：$w_1 \leftarrow w_1 - \eta \frac{\partial \ell}{\partial w_1}$，$b_1$ 同理 |

新参数 $w_1, b_1$ 由 `randn` 随机初始化（与真实参数无关），
训练过程中记录每一轮 epoch 结束时的 $w_1, b_1$ 与损失，用于后续画出参数变化轨迹。

> 约定：训练时用 $\frac{1}{2}(\hat y - y)^2$（除以 2 便于求导），
> 而所有**报告和绘图**中的损失统一用均方误差 MSE $=\frac{1}{n}\sum(\hat y - y)^2$，
> 两者只差常数倍 2，最优解完全相同；这样报告的损失可以直接与噪声方差 $\sigma^2$ 比较。

In [4]:
def linreg(X, w, b):
    """线性模型 y_hat = X w + b；X 形状 (n, 1)，w 形状 (1, 1)，b 形状 (1,)"""
    return X @ w + b


def squared_loss(y_hat, y):
    """训练用的平方损失，除以 2 便于求导"""
    return (y_hat - y) ** 2 / 2


def mse(y_hat, y):
    """报告用的均方误差 MSE（与平方损失只差常数倍 2，不影响最优解）"""
    return ((y_hat - y) ** 2).mean()


def data_iter(X, y, batch_size, seed=None):
    """随机打乱后按 batch_size 划分的小批量迭代器"""
    n = X.shape[0]
    g = torch.Generator().manual_seed(seed) if seed is not None else None
    idx = torch.randperm(n, generator=g)
    for i in range(0, n, batch_size):
        j = idx[i:i + batch_size]
        yield X[j], y[j]


def sgd(params, lr):
    """小批量随机梯度下降：沿负梯度方向更新参数，并清空梯度"""
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad
            p.grad.zero_()

In [5]:
def fit(X, y, w_init, b_init, lr, batch_size, epochs, seed=SEED,
        eval_data=None, snapshot_epochs=()):
    """从零实现线性回归的训练过程。

    eval_data=(X_test, y_test) 时额外记录每轮的测试损失（用于泛化能力评估）；
    snapshot_epochs 中的 epoch 会保存参数快照（用于画出不同轮次的拟合直线）。
    返回：训练后的 (w, b)、逐轮历史 history、参数快照 snapshots
    """
    # 注意形状：w 必须是 (特征数, 1)，否则 X @ w 会丢掉最后一维、与 y 触发错误的广播
    w = torch.tensor([[float(w_init)]], dtype=torch.float32, requires_grad=True)
    b = torch.tensor([float(b_init)], dtype=torch.float32, requires_grad=True)

    history = {"epoch": [], "loss": [], "w": [], "b": []}
    if eval_data is not None:
        history["loss_test"] = []
    snapshots = {}

    for epoch in range(1, epochs + 1):
        # 每一轮都重新打乱数据，遍历所有小批量
        for X_batch, y_batch in data_iter(X, y, batch_size, seed=seed + epoch):
            loss = squared_loss(linreg(X_batch, w, b), y_batch).mean()
            loss.backward()          # 反向传播求梯度
            sgd([w, b], lr)          # 更新参数 w1, b1

        with torch.no_grad():
            # 记录本轮的损失与参数，用于后续画出收敛曲线与参数轨迹
            history["epoch"].append(epoch)
            history["loss"].append(float(mse(linreg(X, w, b), y)))
            history["w"].append(w.item())
            history["b"].append(b.item())
            if eval_data is not None:
                X_test, y_test = eval_data
                history["loss_test"].append(float(mse(linreg(X_test, w, b), y_test)))
        if epoch in snapshot_epochs:
            snapshots[epoch] = (w.item(), b.item())

    # 每轮都应有记录，避免后续绘图拿到空列表
    for key in ("epoch", "loss", "w", "b"):
        assert len(history[key]) == epochs, f"history[{key!r}] 记录不完整"

    return w.detach(), b.detach(), history, snapshots

In [6]:
# 新参数 w1, b1：由 randn 随机初始化，与真实参数 (2.0, 4.2) 相差很远
init = torch.randn(2, generator=torch.Generator().manual_seed(SEED)) * 1.5
W1_INIT, B1_INIT = float(init[0]), float(init[1])

LR, BATCH, EPOCHS = 0.03, 10, 30

print(f"真实参数   w  = {TRUE_W:+.4f}   b  = {TRUE_B:+.4f}")
print(f"初始参数   w1 = {W1_INIT:+.4f}   b1 = {B1_INIT:+.4f}   （randn 随机初始化）")
print(f"超参数     学习率 lr = {LR}，批量大小 = {BATCH}，epoch = {EPOCHS}\n")

w1, b1, hist1, snaps1 = fit(X, y, W1_INIT, B1_INIT, lr=LR, batch_size=BATCH,
                            epochs=EPOCHS, snapshot_epochs=(1, 2, 5, EPOCHS))

print(f"{'epoch':>6}{'训练损失':>14}{'w1':>11}{'b1':>11}")
for ep, l, wv, bv in zip(hist1["epoch"], hist1["loss"], hist1["w"], hist1["b"]):
    if ep in (1, 2, 5, 10, 15, 20, 25, EPOCHS):
        print(f"{ep:>6}{l:>14.5f}{wv:>11.4f}{bv:>11.4f}")

print(f"\n训练后参数 w1 = {w1.item():+.4f}   b1 = {b1.item():+.4f}")
print(f"参数误差   |w1 - w| = {abs(w1.item() - TRUE_W):.4f}    "
      f"|b1 - b| = {abs(b1.item() - TRUE_B):.4f}")
print(f"损失下降   {hist1['loss'][0]:.4f}  ->  {hist1['loss'][-1]:.4f}"
      f"（MSE 的下限是噪声方差 σ² = {NOISE_STD ** 2:.4f}，无法继续下降）")

真实参数   w  = +2.0000   b  = +4.2000
初始参数   w1 = +0.5050   b1 = +0.1932   （randn 随机初始化）
超参数     学习率 lr = 0.03，批量大小 = 10，epoch = 30

 epoch          训练损失         w1         b1
     1       9.86074     0.7143     1.2269
     2       5.76242     0.9017     1.9922
     5       1.42015     1.3436     3.2861
    10       0.43886     1.7567     3.9795
    15       0.35112     1.9408     4.1457
    20       0.34071     2.0181     4.1838
    25       0.33923     2.0482     4.1927
    30       0.33894     2.0624     4.1963

训练后参数 w1 = +2.0624   b1 = +4.1963
参数误差   |w1 - w| = 0.0624    |b1 - b| = 0.0037
损失下降   9.8607  ->  0.3389（MSE 的下限是噪声方差 σ² = 0.3600，无法继续下降）


In [7]:
# 收敛过程：损失曲线 + 两个参数随 epoch 的变化
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=("训练损失（对数纵轴）", "w 的估计值 w1", "b 的估计值 b1"))

fig.add_trace(go.Scatter(
    x=hist1["epoch"], y=hist1["loss"], mode="lines+markers", name="训练损失",
    line=dict(color=C_BLUE, width=2), marker=dict(size=6),
    hovertemplate="epoch %{x}<br>损失 %{y:.5f}<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=hist1["epoch"], y=hist1["w"], mode="lines+markers", name="w1（估计值）",
    line=dict(color=C_BLUE, width=2), marker=dict(size=6),
    hovertemplate="epoch %{x}<br>w1 %{y:.4f}<extra></extra>",
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=hist1["epoch"], y=hist1["b"], mode="lines+markers", name="b1（估计值）",
    line=dict(color=C_BLUE, width=2), marker=dict(size=6), showlegend=False,
    hovertemplate="epoch %{x}<br>b1 %{y:.4f}<extra></extra>",
), row=1, col=3)

# 真实参数作为参考线直接标注（颜色之外的第二重编码：虚线 + 文字）
for col, true_v, lab in ((2, TRUE_W, f"真实 w = {TRUE_W}"), (3, TRUE_B, f"真实 b = {TRUE_B}")):
    fig.add_hline(y=true_v, line=dict(color=C_TRUE, width=1.6, dash="dash"), row=1, col=col)
    fig.add_annotation(x=hist1["epoch"][0], y=true_v, row=1, col=col, text=lab,
                       showarrow=False, xanchor="left", yanchor="bottom",
                       font=dict(color=C_MUTED, size=11))

fig.add_annotation(x=hist1["epoch"][-1], y=hist1["loss"][-1], row=1, col=1,
                   text=f"最终 {hist1['loss'][-1]:.4f}", showarrow=False,
                   xanchor="right", yanchor="top", font=dict(color=C_MUTED, size=11))

fig.update_yaxes(type="log", dtick=1, row=1, col=1)      # 对数轴只保留 10 的整数次幂刻度
fig.update_xaxes(title_text="epoch", row=1, col=1)
fig.update_xaxes(title_text="epoch", row=1, col=2)
fig.update_xaxes(title_text="epoch", row=1, col=3)
fig.update_yaxes(title_text="MSE", row=1, col=1)
fig.update_yaxes(title_text="w1", row=1, col=2)
fig.update_yaxes(title_text="b1", row=1, col=3)
style(fig, "训练收敛过程：损失逐步下降，w1 与 b1 不断逼近真实参数", "", "", width=1080, height=430)
fig.show()

In [8]:
# 拟合效果：不同 epoch 的参数快照所对应的拟合直线
eps = sorted(snaps1)
fig = make_subplots(rows=1, cols=len(eps),
                    subplot_titles=[f"epoch {e}   w1={snaps1[e][0]:.2f}, b1={snaps1[e][1]:.2f}"
                                    for e in eps])

for k, e in enumerate(eps, start=1):
    wv, bv = snaps1[e]
    fig.add_trace(go.Scatter(x=Xn, y=yn, mode="markers", showlegend=False,
                             marker=dict(size=5, color=C_DATA, opacity=0.45),
                             hoverinfo="skip"), row=1, col=k)
    xs, ys = line_xy(Xn, TRUE_W, TRUE_B)
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", showlegend=False,
                             line=dict(color=C_TRUE, width=1.6, dash="dash"),
                             hoverinfo="skip"), row=1, col=k)
    xs, ys = line_xy(Xn, wv, bv)
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", showlegend=False,
                             line=dict(color=C_BLUE, width=2.5),
                             hovertemplate="拟合直线: y = %{y:.2f}<extra></extra>"), row=1, col=k)

fig.add_annotation(x=0.01, y=-0.22, xref="paper", yref="paper", showarrow=False,
                   text="灰色散点 = 观测样本　黑色虚线 = 真实直线　蓝色实线 = 当前拟合直线",
                   font=dict(color=C_MUTED, size=11), xanchor="left")
style(fig, "参数快照对应的拟合效果（训练初期偏差大，后期几乎与真实直线重合）",
      "", "", width=1120, height=380, legend_bottom=False)
fig.show()

In [9]:
# 参数变化轨迹：背景为数据集上损失函数的等高线，红线为 SGD 走过的路径
path_w = np.array(hist1["w"])
path_b = np.array(hist1["b"])

grid_n = 240
w_axis = np.linspace(min(path_w.min(), TRUE_W) - 0.8, max(path_w.max(), TRUE_W) + 0.8, grid_n)
b_axis = np.linspace(min(path_b.min(), TRUE_B) - 1.5, max(path_b.max(), TRUE_B) + 1.5, grid_n)
Wg, Bg = np.meshgrid(w_axis, b_axis)                     # 参数平面上的网格
Z = ((Xn[None, :] * Wg.reshape(-1, 1) + Bg.reshape(-1, 1) - yn[None, :]) ** 2).mean(axis=1)
Z = Z.reshape(Wg.shape)                                  # 每个 (w, b) 组合对应的 MSE

fig = go.Figure()
fig.add_trace(go.Contour(
    x=w_axis, y=b_axis, z=np.log10(Z), colorscale=BLUES, ncontours=30, showlegend=False,
    contours=dict(coloring="heatmap", showlines=False),
    colorbar=dict(title="log10 MSE", thickness=14, len=0.85),
    hovertemplate="w=%{x:.2f}  b=%{y:.2f}<br>log10 MSE=%{z:.2f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=path_w, y=path_b, mode="lines+markers", name="SGD 参数轨迹 (w1, b1)",
    line=dict(color=C_RED, width=2),
    marker=dict(size=7, color="white", line=dict(color=C_RED, width=1.6)),
    text=[f"epoch {e}" for e in hist1["epoch"]],
    hovertemplate="%{text}<br>w1=%{x:.3f}  b1=%{y:.3f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=[path_w[0]], y=[path_b[0]], mode="markers", name="初始参数",
    marker=dict(symbol="square", size=12, color=C_TRUE, line=dict(color="#fcfcfb", width=1)),
    hovertemplate="初始 (w1, b1) = (%{x:.3f}, %{y:.3f})<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=[TRUE_W], y=[TRUE_B], mode="markers", name="真实参数 (最优点)",
    marker=dict(symbol="star", size=17, color=C_RED, line=dict(color="#fcfcfb", width=1)),
    hovertemplate="真实 (w, b) = (%{x:.2f}, %{y:.2f})<extra></extra>",
))
# 直接用带底色的标注指向两个关键点，避免文字压在等高线上看不清
note = dict(showarrow=True, arrowhead=0, arrowwidth=1, arrowcolor=C_MUTED,
            bgcolor="rgba(252,252,251,0.92)", bordercolor="#d8d7d2", borderwidth=1,
            borderpad=3, font=dict(color=C_MUTED, size=11))
fig.add_annotation(x=path_w[0], y=path_b[0], ax=75, ay=-35,
                   text=f"初始 (w1, b1) = ({path_w[0]:.2f}, {path_b[0]:.2f})", **note)
fig.add_annotation(x=TRUE_W, y=TRUE_B, ax=95, ay=45,
                   text=f"真实 (w, b) = ({TRUE_W}, {TRUE_B})", **note)
style(fig, "参数空间中的收敛轨迹（背景为损失等高线）", "w", "b", width=780, height=540)
fig.show()

**第二部分观察到的现象**

* 损失在最初几轮急剧下降，随后趋于平缓——损失曲面是凸的，梯度下降很快进入最优解附近的“平底”区域；
* $w_1, b_1$ 从随机初值出发，沿梯度反方向（与等高线近似垂直）逐步逼近真实参数，轨迹在 $(w, b)$ 平面上是一段折线而非直线；
* 参数误差同时受**学习率**（步子大小）与**噪声**（$\sigma^2$）影响：损失最终停留在 $\sigma^2$ 附近而不是 0。

## 三、扩展数据规模与训练/测试集划分：泛化能力评估

上面用 100 条样本训练、又在同样的 100 条样本上评估，只能说明“拟合得好”，
并不能说明模型对**没见过的数据**是否同样有效。下面把数据规模扩大到 1000 条，
按 7:3 划分为训练集与测试集：

* **训练集**：用于更新参数 $w_1, b_1$；
* **测试集**：训练全程不参与梯度计算，只用于评估，衡量**泛化能力**。

In [10]:
N_ALL, TEST_RATIO = 1000, 0.3
X_all, y_all = synthetic_data(TRUE_W, TRUE_B, N_ALL, seed=2024)

perm = torch.randperm(N_ALL, generator=torch.Generator().manual_seed(SEED)).numpy()
n_test = int(N_ALL * TEST_RATIO)
idx_test, idx_train = perm[:n_test], perm[n_test:]

X_train, y_train = X_all[idx_train], y_all[idx_train]
X_test, y_test = X_all[idx_test], y_all[idx_test]

print(f"数据规模：{N_ALL} 条（第二部分为 {N} 条），按 {1 - TEST_RATIO:.0%} : {TEST_RATIO:.0%} 划分")
print(f"训练集 {X_train.shape[0]} 条，测试集 {X_test.shape[0]} 条")
print(f"训练集 y 均值 {y_train.mean():+.3f}，测试集 y 均值 {y_test.mean():+.3f}（同分布）")

数据规模：1000 条（第二部分为 100 条），按 70% : 30% 划分
训练集 700 条，测试集 300 条
训练集 y 均值 +4.197，测试集 y 均值 +4.202（同分布）


In [11]:
# 同一套超参数，在训练集上训练，同时记录训练损失与测试损失
w2, b2, hist2, _ = fit(X_train, y_train, W1_INIT, B1_INIT, lr=LR, batch_size=BATCH,
                       epochs=EPOCHS, eval_data=(X_test, y_test))

print(f"{'epoch':>6}{'训练损失':>14}{'测试损失':>14}{'w1':>11}{'b1':>11}")
for ep, ltr, lte, wv, bv in zip(hist2["epoch"], hist2["loss"], hist2["loss_test"],
                                hist2["w"], hist2["b"]):
    if ep in (1, 2, 5, 10, 20, 30):
        print(f"{ep:>6}{ltr:>14.5f}{lte:>14.5f}{wv:>11.4f}{bv:>11.4f}")

gap = hist2["loss_test"][-1] - hist2["loss"][-1]
print(f"\n训练后参数 w1 = {w2.item():+.4f}   b1 = {b2.item():+.4f}")
print(f"参数误差   |w1 - w| = {abs(w2.item() - TRUE_W):.4f}    "
      f"|b1 - b| = {abs(b2.item() - TRUE_B):.4f}")
print(f"最终训练损失 {hist2['loss'][-1]:.4f}，最终测试损失 {hist2['loss_test'][-1]:.4f}，"
      f"泛化间隙 {gap:+.4f}")

 epoch          训练损失          测试损失         w1         b1
     1       0.59415       0.64399     1.8644     3.6948
     2       0.35741       0.37748     1.9928     4.1140
     5       0.35514       0.37239     2.0192     4.1545
    10       0.35678       0.37394     1.9781     4.1419
    20       0.35542       0.37401     2.0243     4.1409
    30       0.35534       0.37369     2.0235     4.1434

训练后参数 w1 = +2.0235   b1 = +4.1434
参数误差   |w1 - w| = 0.0235    |b1 - b| = 0.0566
最终训练损失 0.3553，最终测试损失 0.3737，泛化间隙 +0.0183


In [12]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5],
                    subplot_titles=("训练损失与测试损失（MSE）", "模型在测试集上的拟合效果"))

fig.add_trace(go.Scatter(
    x=hist2["epoch"], y=hist2["loss"], mode="lines+markers", name="训练损失（700 条）",
    line=dict(color=C_BLUE, width=2), marker=dict(size=6),
    hovertemplate="epoch %{x}<br>训练损失 %{y:.5f}<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=hist2["epoch"], y=hist2["loss_test"], mode="lines+markers", name="测试损失（300 条）",
    line=dict(color=C_ORANGE, width=2), marker=dict(size=6),
    hovertemplate="epoch %{x}<br>测试损失 %{y:.5f}<extra></extra>",
), row=1, col=1)
fig.add_hline(y=NOISE_STD ** 2, line=dict(color=C_TRUE, width=1.4, dash="dash"), row=1, col=1)
fig.add_annotation(x=hist2["epoch"][0], y=NOISE_STD ** 2 - 0.04, row=1, col=1,
                   text=f"噪声方差 σ² = {NOISE_STD ** 2:.2f}", showarrow=False,
                   xanchor="left", yanchor="middle", font=dict(color=C_MUTED, size=11))
fig.add_annotation(x=hist2["epoch"][-1], y=hist2["loss_test"][-1], row=1, col=1,
                   text=f"{hist2['loss_test'][-1]:.4f}", showarrow=False,
                   xanchor="right", yanchor="bottom", font=dict(color=C_MUTED, size=11))

Xtn, ytn = X_test.flatten().numpy(), y_test.flatten().numpy()
fig.add_trace(go.Scatter(
    x=Xtn, y=ytn, mode="markers", name="测试集样本",
    marker=dict(size=6, color=C_DATA, opacity=0.6),
    hovertemplate="x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>",
), row=1, col=2)
xs, ys = line_xy(Xtn, TRUE_W, TRUE_B)
fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", name="真实直线", showlegend=False,
                         line=dict(color=C_TRUE, width=1.6, dash="dash"), hoverinfo="skip"),
              row=1, col=2)
xs, ys = line_xy(Xtn, w2.item(), b2.item())
fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", name=f"拟合直线 y = {w2.item():.3f}x + {b2.item():.3f}",
                         line=dict(color=C_BLUE, width=2.5), hoverinfo="skip"), row=1, col=2)

fig.update_yaxes(range=[0.28, 0.70], dtick=0.1, row=1, col=1)   # 下方留白放 σ² 参考线标注
fig.update_xaxes(title_text="epoch", row=1, col=1)
fig.update_yaxes(title_text="MSE", row=1, col=1)
fig.update_xaxes(title_text="x", row=1, col=2)
fig.update_yaxes(title_text="y", row=1, col=2)
style(fig, "泛化评估：训练损失与测试损失同步下降并趋于一致", "", "", width=1120, height=470)
fig.show()

In [13]:
def evaluate(w, b, X_eval, y_eval):
    """返回 (MSE, R²)：R² = 1 - SS_res / SS_tot，越接近 1 说明拟合越好"""
    with torch.no_grad():
        pred = linreg(X_eval, torch.tensor([[float(w)]]), torch.tensor([float(b)]))
        loss = float(mse(pred, y_eval))
        r2 = float(1 - ((y_eval - pred) ** 2).sum() / ((y_eval - y_eval.mean()) ** 2).sum())
    return loss, r2


# 小样本模型：只用训练集的前 100 条训练，与 700 条训练做对照
w_small, b_small, hist_small, _ = fit(X_train[:100], y_train[:100], W1_INIT, B1_INIT,
                                      lr=LR, batch_size=BATCH, epochs=EPOCHS)

rows = []
cases = [
    ("模型A：训练集前 100 条", 100, w_small, b_small, X_train[:100], y_train[:100]),
    ("模型B：全部训练集 700 条", 700, w2, b2, X_train, y_train),
    ("对照：第二部分模型（另生成的 100 条）", 100, w1, b1, X, y),
]
for name, n, wt, bt, X_tr, y_tr in cases:
    mse_tr, _ = evaluate(wt, bt, X_tr, y_tr)
    mse_te, r2_te = evaluate(wt, bt, X_test, y_test)     # 统一在同一测试集上评估
    rows.append({
        "训练样本数": n,
        "w1": round(float(wt), 3),
        "b1": round(float(bt), 3),
        "训练损失": round(mse_tr, 4),
        "测试损失": round(mse_te, 4),
        "测试 R²": round(r2_te, 4),
        "泛化间隙": round(mse_te - mse_tr, 4),
    })

df_metrics = pd.DataFrame(rows, index=[c[0] for c in cases])
df_metrics

,训练样本数,w1,b1,训练损失,测试损失,测试 R²,泛化间隙
模型A：训练集前 100 条,100,2.003,4.186,0.3932,0.3702,0.9144,-0.0230
模型B：全部训练集 700 条,700,2.023,4.143,0.3553,0.3737,0.9136,0.0183
对照：第二部分模型（另生成的 100 条）,100,2.062,4.196,0.3389,0.3737,0.9136,0.0347


In [14]:
# 数据规模对泛化误差的影响：逐步增大训练样本数，在同一测试集上评估
sizes = [20, 50, 100, 200, 400, 700]
size_rows = []
for n in sizes:
    w_n, b_n, _, _ = fit(X_train[:n], y_train[:n], W1_INIT, B1_INIT,
                         lr=LR, batch_size=BATCH, epochs=EPOCHS)
    mse_tr, _ = evaluate(w_n, b_n, X_train[:n], y_train[:n])
    mse_te, r2_te = evaluate(w_n, b_n, X_test, y_test)
    size_rows.append({"训练样本数": n, "训练损失": round(mse_tr, 4),
                      "测试损失": round(mse_te, 4), "测试 R²": round(r2_te, 4)})

df_size = pd.DataFrame(size_rows)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_size["训练样本数"], y=df_size["测试损失"], mode="lines+markers+text", name="测试损失",
    line=dict(color=C_BLUE, width=2),
    marker=dict(size=9, color="white", line=dict(color=C_BLUE, width=2)),
    text=["" if i not in (0, len(df_size) - 1) else f"{v:.4f}"
          for i, v in enumerate(df_size["测试损失"])],
    textposition="top center", textfont=dict(color=C_MUTED, size=11),
    hovertemplate="训练样本 %{x} 条<br>测试损失 %{y:.4f}<extra></extra>",
))
fig.add_hline(y=NOISE_STD ** 2, line=dict(color=C_TRUE, width=1.6, dash="dash"),
              annotation_text=f"噪声方差 σ² = {NOISE_STD ** 2:.2f}（不可约误差）",
              annotation_position="bottom right", annotation_font=dict(color=C_MUTED, size=11))
style(fig, "数据规模与泛化误差：样本越多，测试损失越接近噪声方差", "训练样本数", "测试损失（均值）",
      width=780, height=460)
fig.show()

## 四、实验小结

1. **建模**：由 $y = wx + b + \mathrm{randn}$ 生成的 $\{x, y\}$ 满足线性关系叠加高斯噪声，
   用同一个函数形式 $y = w_1 x + b_1$ 建模，模型是**可识别**的，理论上可以恢复出真实参数。
2. **收敛过程**：$w_1, b_1$ 由 `randn` 随机初始化，经小批量随机梯度下降后沿损失等高线的法向逼近真实参数；
   损失曲线先陡后缓，参数轨迹在 $(w, b)$ 平面上是折线，学习率决定步长与收敛速度。
3. **拟合效果**：训练足够轮次后，拟合直线与真实直线几乎重合，
   但训练损失不会降到 0，而是稳定在噪声方差 $\sigma^2$ 附近——这部分是数据本身决定的**不可约误差**。
4. **泛化能力**：测试集不参与训练，其损失曲线与训练损失曲线几乎重合且同步下降，
   说明模型没有过拟合；样本量越大，测试损失越接近 $\sigma^2$，泛化间隙越小，
   $R^2$ 越接近噪声决定的上限。

In [15]:
print("=" * 62)
print("实验结论（关键数值）")
print("=" * 62)
print(f"真实参数：w = {TRUE_W:+.4f}, b = {TRUE_B:+.4f}；噪声方差 σ² = {NOISE_STD ** 2:.4f}")
print("-" * 62)
print(f"[第二部分] {N} 条数据训练 {EPOCHS} 轮")
print(f"  初始参数   w1 = {W1_INIT:+.4f}, b1 = {B1_INIT:+.4f}")
print(f"  训练后参数 w1 = {w1.item():+.4f}, b1 = {b1.item():+.4f}")
print(f"  参数误差   |Δw| = {abs(w1.item() - TRUE_W):.4f}, |Δb| = {abs(b1.item() - TRUE_B):.4f}")
print(f"  训练损失   {hist1['loss'][0]:.4f} -> {hist1['loss'][-1]:.4f}")
print("-" * 62)
print(f"[第三部分] {N_ALL} 条数据，{X_train.shape[0]}/{X_test.shape[0]} 划分，训练 {EPOCHS} 轮")
print(f"  训练后参数 w1 = {w2.item():+.4f}, b1 = {b2.item():+.4f}")
mse_te7, r2_te7 = evaluate(w2, b2, X_test, y_test)
print(f"  训练损失 {hist2['loss'][-1]:.4f}，测试损失 {hist2['loss_test'][-1]:.4f}，"
      f"泛化间隙 {hist2['loss_test'][-1] - hist2['loss'][-1]:+.4f}，测试 R² = {r2_te7:.4f}")
print("-" * 62)
print("[数据规模] 同一测试集上的泛化误差")
for _, r in df_size.iterrows():
    print(f"  训练样本 {int(r['训练样本数']):>4} 条 -> 测试损失 {r['测试损失']:.4f}，测试 R² {r['测试 R²']:.4f}")
print(f"  泛化误差下限 = 噪声方差 σ² = {NOISE_STD ** 2:.4f}")
print("=" * 62)

实验结论（关键数值）
真实参数：w = +2.0000, b = +4.2000；噪声方差 σ² = 0.3600
--------------------------------------------------------------
[第二部分] 100 条数据训练 30 轮
  初始参数   w1 = +0.5050, b1 = +0.1932
  训练后参数 w1 = +2.0624, b1 = +4.1963
  参数误差   |Δw| = 0.0624, |Δb| = 0.0037
  训练损失   9.8607 -> 0.3389
--------------------------------------------------------------
[第三部分] 1000 条数据，700/300 划分，训练 30 轮
  训练后参数 w1 = +2.0235, b1 = +4.1434
  训练损失 0.3553，测试损失 0.3737，泛化间隙 +0.0183，测试 R² = 0.9136
--------------------------------------------------------------
[数据规模] 同一测试集上的泛化误差
  训练样本   20 条 -> 测试损失 0.8377，测试 R² 0.8062
  训练样本   50 条 -> 测试损失 0.3764，测试 R² 0.9129
  训练样本  100 条 -> 测试损失 0.3702，测试 R² 0.9144
  训练样本  200 条 -> 测试损失 0.3749，测试 R² 0.9133
  训练样本  400 条 -> 测试损失 0.3752，测试 R² 0.9132
  训练样本  700 条 -> 测试损失 0.3737，测试 R² 0.9136
  泛化误差下限 = 噪声方差 σ² = 0.3600
